# Gram-stain smear calls that generalise to new laboratories and species

`ov.tl.smear_*` measures every boxed object in bright-field images of stained smears (bacteria,
yeasts, ...), summarises the objects of each image (a *field*), and calls one label per field with a
**two-axis classifier**: a morphology stage (stain-free shape features) and a stain stage
(two-dye colour), joined by a prior P(stain | shape).

This tutorial covers the full workflow on synthetic images, then the section
**"Generalising to new laboratories and species"** shows the two options that control how the
classifier behaves on species groups and staining protocols it never saw in training:

* `stain_feature` - which colour measurements feed the stain stage;
* `prior` - how much the shape -> stain association learned from the training species is trusted
  (`"group"`, `"tempered"`, `"flat"`, or `"em"` batch label-shift adaptation).

All images below are synthetic (generated in the notebook), so the tutorial runs in about a minute
on a laptop.

In [1]:
import numpy as np
import pandas as pd
import omicverse as ov
from omicverse.tl import _smear_morph_core as smc
print(ov.__version__)

2.3.2


## 1. A synthetic smear generator

Each field holds objects of one culture: round cells in short chains (cocci), rods, or large ovals
(yeast). The colour of every pixel is a Beer-Lambert mixture of a violet primary dye and a pink
counterstain. A species has its own **decolourisation level**: Gram-positive species keep mostly
violet, Gram-negative ones mostly pink, but some species are "Gram-variable" and retain an
intermediate mix. A laboratory has its own **counterstain hue, stain strength and background**.

In [2]:
VIOLET = np.array([0.90, 0.39, 0.19])          # optical-density directions of the two dyes
PINK_LAB = {"lab_A": np.array([0.06, 0.97, 0.21]),   # safranin-like
            "lab_B": np.array([0.20, 0.95, 0.25])}   # a bluer (fuchsin-like) counterstain

def unit(v):
    return v / np.linalg.norm(v)

def make_field(rng, shape, violet_share, lab="lab_A", strength=0.8, n=8, size=220):
    pink = unit(PINK_LAB[lab])
    bg = 240.0 if lab == "lab_A" else 228.0
    img = np.full((size, size, 3), bg) + rng.normal(0, 2.0, (size, size, 3))
    yy, xx = np.mgrid[:size, :size]
    boxes = []
    for k in range(n):
        cx, cy = 30 + (k % 4) * 50, 35 + (k // 4) * 95
        if shape == "round":
            m = np.zeros((size, size), bool)
            for j in range(3):
                m |= (xx - (cx + 8 * j)) ** 2 + (yy - cy) ** 2 <= 16
        elif shape == "rod":
            m = (np.abs(xx - cx - 8) <= 12) & (np.abs(yy - cy) <= 3.5)
        else:
            m = ((xx - cx - 8) / 14.0) ** 2 + ((yy - cy) / 10.0) ** 2 <= 1
        share = np.clip(violet_share + rng.normal(0, 0.05), 0, 1)
        od = strength * rng.uniform(0.8, 1.2) * (share * unit(VIOLET) + (1 - share) * pink)
        img[m] = bg * 10 ** (-od)
        ys, xs = np.nonzero(m)
        boxes.append((xs.min() - 1, ys.min() - 1, xs.max() + 1, ys.max() + 1))
    return np.clip(img, 0, 255), boxes

## 2. A development cohort: tables in, objects measured

`smear_measure` takes two tables (fields with image path and pixel size, objects with boxes) and
returns the per-object table, the per-field summary table and the dye vectors it estimated from the
cohort itself (label-free non-negative factorisation of the object optical densities).

As in many real collections, the rods of the development cohort are mostly Gram-negative species,
and only one Gram-positive rod species is present.

In [3]:
import os, tempfile
from PIL import Image

AXES = {"gp_cocci": ("round", "pos"), "gp_rods": ("rod", "pos"),
        "gn_rods": ("rod", "neg"), "gn_cocci": ("round", "neg"), "yeast": ("big", None)}

def make_cohort(root, species, lab, per=4, seed=0):
    rng = np.random.default_rng(seed)
    os.makedirs(f"{root}/fields", exist_ok=True)
    F, O, L = [], [], []
    for sp, (label, violet_share, strength) in species.items():
        shape = AXES[label][0]
        for k in range(per):
            fid = f"{lab}_{sp}_{k}"
            img, boxes = make_field(rng, shape, violet_share, lab=lab, strength=strength)
            Image.fromarray(img.astype(np.uint8)).save(f"{root}/fields/{fid}.png")
            F.append({"field_id": fid, "image_path": f"fields/{fid}.png", "um_per_px": 0.1})
            L.append({"field_id": fid, "culture_class": label, "group_id": sp})
            O += [{"organism_id": f"{fid}_{j}", "field_id": fid, "x_min": b[0], "y_min": b[1],
                   "x_max": b[2], "y_max": b[3]} for j, b in enumerate(boxes)]
    return pd.DataFrame(F), pd.DataFrame(O), pd.DataFrame(L)

dev_species = {  # species: (class, violet share retained, stain strength)
    "sp_c1": ("gp_cocci", 0.90, 0.8), "sp_c2": ("gp_cocci", 0.80, 0.9), "sp_c3": ("gp_cocci", 0.85, 0.7),
    "sp_r1": ("gp_rods", 0.70, 0.8),
    "sp_n1": ("gn_rods", 0.15, 0.8), "sp_n2": ("gn_rods", 0.30, 0.7), "sp_n3": ("gn_rods", 0.10, 0.9),
    "sp_n4": ("gn_rods", 0.35, 0.8), "sp_y1": ("yeast", 0.80, 1.0), "sp_y2": ("yeast", 0.60, 1.0),
}
dev_root = tempfile.mkdtemp(prefix="smear_dev_")
fields, objects, labels = make_cohort(dev_root, dev_species, "lab_A", seed=1)
objs, X, vec = ov.tl.smear_measure(fields, objects, root=dev_root)
print(X.shape, "dye vectors:\n", np.round(vec, 3))
objs[["organism_id", "long_axis_um", "short_axis_um", "stain_index",
      "stain_core_primary_fraction", "stain_core_hsd_cx"]].head()

(40, 346) dye vectors:
 [[0.886 0.419 0.2  ]
 [0.078 0.973 0.218]]


,organism_id,long_axis_um,short_axis_um,stain_index,stain_core_primary_fraction,stain_core_hsd_cx
0,lab_A_sp_c1_0_0,2.694754,0.789473,0.941917,0.942128,0.690201
1,lab_A_sp_c1_0_1,2.694754,0.789473,0.890831,0.890983,0.615766
2,lab_A_sp_c1_0_2,2.694754,0.789473,0.999737,1.000000,0.799817
3,lab_A_sp_c1_0_3,2.694754,0.789473,0.952307,0.952658,0.703406
4,lab_A_sp_c1_0_4,2.694754,0.789473,0.928693,0.928982,0.671171


## 3. Fit and validate leave-one-species-out

`smear_fit_predictor` measures, validates by leaving one group (species) out at a time and fits on
all groups. The report gives the recall of each class averaged over its held-out species.
A class represented by a single species (here the Gram-positive rods) cannot be recalled when that
species is held out: its label is never emitted by a model that did not see it.

In [4]:
model, rep = ov.tl.smear_fit_predictor(fields, objects, labels, label_col="culture_class",
                                       group_col="group_id", label_axes=AXES,
                                       positive_stain="pos", root=dev_root)
print("defaults:", rep["stain_feature"], "/", rep["prior"])
print({k: round(v, 2) for k, v in rep["logo_species_balanced_recall"].items()},
      "macro", round(rep["logo_macro"], 3))

defaults: core / tempered
{'gn_rods': 1.0, 'gp_cocci': 1.0, 'gp_rods': 0.0, 'yeast': 1.0} macro 0.75


## 4. Generalising to new laboratories and species

Two things go wrong when the fitted model meets another laboratory:

1. **Colour.** The counterstain hue, stain strength and background differ, and objects can be dense or
   sit on stained debris, so whole-object colour averages drift. The `stain_core_*` measurements use
   only the densest half of each object's pixels (the cell body), are ratios (independent of how much
   dye there is), and come in a background-subtracted version (`*_bgsub`). They follow colour
   deconvolution (Ruifrok & Johnston 2001) and the hue-saturation-density chromaticity of
   van der Laak et al. (2000).
2. **Prior.** `prior="group"` multiplies the colour evidence by P(stain | shape) counted over the
   training species. If almost every training rod is Gram-negative, a *new* Gram-positive rod species
   needs overwhelming colour evidence to be called positive. `prior="tempered"` keeps the direction
   but halves its log-odds (`prior_weight=0.5`); `prior="flat"` drops it; `prior="em"` starts from the
   tempered prior and re-estimates P(stain | shape) on the batch being predicted with the EM
   procedure of Saerens, Latinne & Decaestecker (2002), so a batch full of positive rods moves the
   prior towards positive.

The new laboratory below uses a bluer counterstain, darker background, weaker staining, and three
Gram-positive rod species never seen in training (one of them Gram-variable).

In [5]:
new_species = {
    "new_r1": ("gp_rods", 0.50, 0.6), "new_r2": ("gp_rods", 0.45, 0.6), "new_r3": ("gp_rods", 0.55, 0.5),
    "new_n1": ("gn_rods", 0.25, 0.6), "new_n2": ("gn_rods", 0.15, 0.6), "new_c1": ("gp_cocci", 0.80, 0.6), "new_y1": ("yeast", 0.7, 0.9),
}
new_root = tempfile.mkdtemp(prefix="smear_new_")
f_new, o_new, l_new = make_cohort(new_root, new_species, "lab_B", seed=2)
_, X_new, _ = ov.tl.smear_measure(f_new, o_new, root=new_root)   # dye vectors re-estimated here
X_new = X_new.merge(l_new, on="field_id")
Xdev = X.merge(labels, on="field_id")
feats = [c for c in X.columns if c != "field_id"]

Compare the colour sets and prior strategies: train on the development cohort, predict the new
laboratory in one batch, and score the Gram call (positive / negative) per species.

In [6]:
def gram(lbls):
    return np.array([AXES[l][1] for l in lbls], dtype=object)

rows = []
for sf in ["all", "fraction", "core"]:
    for pr in ["group", "tempered", "flat", "em"]:
        m = ov.tl.two_axis_classifier_fit(Xdev[feats], Xdev.culture_class, Xdev.group_id, AXES,
                                          positive_stain="pos", stain_feature=sf, prior=pr)
        pred, info = ov.tl.two_axis_classifier_predict(m, X_new[feats], return_scores=True)
        keep = X_new.culture_class != "yeast"
        per, macro = ov.tl.species_balanced_recall(gram(X_new.culture_class[keep]),
                                                   gram(pred[keep.to_numpy()]), X_new.group_id[keep])
        rows.append({"stain_feature": sf, "prior": pr, "gram_pos": per["pos"], "gram_neg": per["neg"],
                     "gram_macro": macro,
                     "P(pos|rod) used": 1 / (1 + np.exp(-info["stain_prior_logodds"]["rod"]))})
pd.DataFrame(rows).round(3)

,stain_feature,prior,gram_pos,gram_neg,gram_macro,P(pos|rod) used
0,all,group,0.875,1.0,0.938,0.286
1,all,tempered,0.875,1.0,0.938,0.387
2,all,flat,0.875,1.0,0.938,0.500
3,all,em,0.875,1.0,0.938,0.509
4,fraction,group,0.500,1.0,0.750,0.286
5,fraction,tempered,0.625,1.0,0.812,0.387
6,fraction,flat,0.625,1.0,0.812,0.500
7,fraction,em,0.500,1.0,0.750,0.256
8,core,group,0.688,1.0,0.844,0.286
9,core,tempered,0.688,1.0,0.844,0.387


On this synthetic laboratory the whole-object `all` set happens to do best and the priors matter
less than the colour set: no option wins everywhere, so the library defaults were chosen on real
images (table further below), not on this toy example.

`P(pos|rod) used` is the prior actually applied: about 0.3 for the group prior (one Gram-positive rod
species out of five rod species, Laplace-smoothed), 0.5 for `flat`, and for `em` whatever the new
batch supports.

How the prior was adapted can be inspected directly with `em_stain_prior`: given the calibrated stain
log likelihood ratios and the shape probabilities of a batch, it returns the batch prior.

In [7]:
m = ov.tl.two_axis_classifier_fit(Xdev[feats], Xdev.culture_class, Xdev.group_id, AXES,
                                  positive_stain="pos", stain_feature="core", prior="em")
_, info = ov.tl.two_axis_classifier_predict(m, X_new[feats], return_scores=True)
rod = info["shape_classes"].index("rod")
p0 = m["prior"]["rod"]["pos"]
print("training P(pos | rod) =", round(p0, 3))
print("EM batch P(pos | rod) =", round(smc.em_stain_prior(info["stain_llr"], info["p_shape"][:, rod],
                                                          1 / (1 + np.exp(-0.5 * np.log(p0 / (1 - p0))))), 3))

training P(pos | rod) = 0.286
EM batch P(pos | rod) = 0.402


### Real-data comparison behind the defaults

The same comparison was run on real images: the development Gram-stain data of the omicverse task
(17 species groups, leave-one-species-out and 4-fold group splits) and two openly licensed public
data sets, the positive-blood-culture Gram stains of figshare 26004610 (505 images, 57 species,
CC BY 4.0; Sci Data 2026, doi:10.1038/s41597-026-06651-3) and the M-ROSE lower-respiratory Gram stains
of zenodo 10526360 (Clinical Bacteria DataSet, CC BY 4.0; per-box Gram/shape labels). The tables are
in the pull request that introduced these options; the defaults below were chosen from them.

| stain_feature / prior | dev LOSO macro | dev LOSO Gram macro | dev worst class | dev worst species | blood culture (dev -> PBC) Gram macro | PBC species < 50 % Gram | PBC -> dev Gram macro | respiratory (dev -> M-ROSE) Gram macro |
|---|---|---|---|---|---|---|---|---|
| all / group (original) | 0.859 | 0.863 | 0.704 | 0.40 | 0.846 | 7 / 50 | 0.932 | 0.600 |
| all / tempered | 0.869 | 0.887 | 0.676 | 0.20 | 0.868 | 5 / 50 | 0.935 | 0.614 |
| all / em | 0.852 | 0.865 | 0.590 | 0.00 | 0.868 | 5 / 50 | 0.942 | 0.625 |
| fraction / group | 0.851 | 0.865 | 0.686 | 0.38 | 0.842 | 5 / 50 | 0.948 | 0.573 |
| core / group | 0.855 | 0.870 | 0.722 | 0.40 | 0.828 | 5 / 50 | 0.940 | 0.535 |
| **core / tempered (default)** | **0.876** | **0.897** | **0.704** | **0.40** | **0.851** | **4 / 50** | **0.940** | 0.555 |
| core / flat | 0.869 | 0.887 | 0.676 | 0.20 | 0.851 | 4 / 50 | 0.930 | 0.575 |
| core / em | 0.884 | 0.913 | 0.704 | 0.40 | 0.841 | 4 / 50 | 0.942 | 0.504 |

Field-level AUC of the median object stain score (Gram-positive vs Gram-negative): whole-object
primary fraction 0.983 / 0.919 / 0.776 (development / blood culture / respiratory), core primary
fraction 0.986 / 0.913 / 0.805, background-subtracted core fraction 0.982 / 0.880 / 0.836. On the
respiratory specimens (1-2 boxes per pseudo-field, mucus and host cells) every model is biased towards
Gram-positive; the core colour separates better but its calibrated threshold transfers worse, and
`em` amplifies that bias - which is why `em` is opt-in.

## 5. Export a stand-alone predictor

`smear_export_predictor` writes `predict.py`, the numpy/scipy/scikit-image/scikit-learn-only core,
the pickled model and a small config. The chosen `stain_feature` / `prior` live in the model, and the
EM prior adaptation runs inside the exported core on the whole input batch.

In [8]:
import shutil
new_in = tempfile.mkdtemp(prefix="smear_in_")
shutil.copytree(f"{new_root}/fields", f"{new_in}/fields")
f_new.to_csv(f"{new_in}/fields.csv", index=False)
o_new.to_csv(f"{new_in}/organisms.csv", index=False)
sub = tempfile.mkdtemp(prefix="smear_sub_")
res = ov.tl.smear_export_predictor(model, sub, check_input_dir=new_in, n_jobs=1)
print({k: res[k] for k in ("check_returncode", "fields_covered", "objects_covered", "all_finite",
                           "labels_emitted")})

{'check_returncode': 0, 'fields_covered': True, 'objects_covered': True, 'all_finite': True, 'labels_emitted': ['gn_rods', 'gp_cocci', 'gp_rods', 'yeast']}


## References

* Ruifrok AC, Johnston DA (2001) Quantification of histochemical staining by color deconvolution.
  *Anal Quant Cytol Histol* 23:291-299.
* van der Laak JA et al. (2000) Hue-saturation-density (HSD) model for stain recognition in digital
  images from transmitted light microscopy. *Cytometry* 39:275-284.
* Macenko M et al. (2009) A method for normalizing histology slides for quantitative analysis. *ISBI*.
* Vahadane A et al. (2016) Structure-preserving color normalization and sparse stain separation for
  histological images. *IEEE TMI* 35:1962-1971.
* Saerens M, Latinne P, Decaestecker C (2002) Adjusting the outputs of a classifier to new a priori
  probabilities: a simple procedure. *Neural Comput* 14:21-41.
* Smith KP, Kang AD, Kirby JE (2018) Automated interpretation of blood culture Gram stains by use of a
  deep convolutional neural network. *J Clin Microbiol* 56:e01521-17.